# День 3 — дообучение бейзлайна организаторов
Старт с весов `baseline.onnx`, дообучение нашим лоссом (= метрика) на 3500 последовательностях с маленьким learning rate.

Перед запуском: **Среда выполнения → Сменить среду выполнения → T4 GPU**. Положите `baseline.onnx` в `Мой диск/wunder/` и обновлённую папку `day2` поверх старой.

## 1. GPU + Drive + код

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader
from google.colab import drive
drive.mount('/content/drive')
W = '/content/drive/MyDrive/wunder'
!pip -q install onnx
!rm -rf /content/day2 && cp -r $W/day2 /content/day2
%cd /content/day2
!ls -la $W/baseline.onnx
!python -m pytest -q tests/test_day2.py -k 'loss or chunked'

## 2. Данные (~12 мин; диск Colab очищается при каждом подключении)

In [ ]:
!python prepare_data.py --src $W/colab_data/valid_sub.parquet --out /content/data/valid
!python prepare_data.py --src $W/colab_data/train_sub.parquet --out /content/data/train

## 3. Дообучение бейзлайна (~35 мин)
Первая строка `VALID (start = initial weights)` — скор исходного бейзлайна на нашей валидации: это планка. Дальше каждая эпоха должна быть выше неё. Если сессия оборвалась: ячейки 1–2, затем эта же ячейка с `--resume`.

In [ ]:
RUN = W + '/runs/ft_baseline_v1'
!python train.py --train-dir /content/data/train --valid-dir /content/data/valid --out $RUN \
    --arch baseline --init-onnx $W/baseline.onnx \
    --epochs 10 --batch 32 --chunk 1000 --lr 3e-4 --mse-weight 0.1

## 4. Экспорт и скачивание

In [ ]:
!python export.py --ckpt $RUN/best.pt --valid-dir /content/data/valid --out $W/submissions/ft_baseline_v1
from google.colab import files
files.download(W + '/submissions/ft_baseline_v1.zip')

## 5. (Пока сабмит считается) наша GRU, но обученная дольше: 25 эпох вместо 6 (~75 мин)
В прошлый раз скор рос до последней эпохи — модель была недоучена.

In [ ]:
RUN2 = W + '/runs/gru128x2_v2_long'
!python train.py --train-dir /content/data/train --valid-dir /content/data/valid --out $RUN2 \
    --epochs 25 --batch 64 --chunk 1000 --lr 2e-3
!python export.py --ckpt $RUN2/best.pt --valid-dir /content/data/valid --out $W/submissions/gru128x2_v2_long